<style>
    h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h2>
    Preparation
</h2>

In [1]:
import requests
import pandas as pd
from datetime import datetime
import calendar
import time

api_key = open("api_key.txt", "r").read()
locations = {
    "London": (51.5074, -0.1278),
    "Kent": (51.2787, 0.5217),
    "Essex": (51.7810, 0.2382),
    "Hertfordshire": (51.7670, -0.2087),
    "Surrey": (51.3148, -0.5590),
    "Buckinghamshire": (51.9827, -0.9844),
    "East Sussex": (50.8930, 0.0553),
    "West Sussex": (50.9280, -0.4610)
}

<style>
    h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h2>
    Turn dates to unix timestamp (number of seconds since 1970-1-1)
</h2>

In [ ]:
def datetime_to_unix(datetime_str):
    datetime_obj = datetime.strptime(datetime_str, "%Y-%m-%d %H:%M")
    unix_timestamp = int(datetime_obj.timestamp())
    return unix_timestamp

<style>
    h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h2>
    Fetch from url with given latitude, longitude, start and end time
</h2>

In [ ]:
def get_air_pollution_data(api_key, lat, lon, start_timestamp, end_timestamp):
    url = f"http://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start_timestamp}&end={end_timestamp}&appid={api_key}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        return None

<style>
    h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h2>
    Collect pollution data between start and end year and save as CSV file
</h2>

In [ ]:
def process_and_save_air_pollution_data(api_key, locations, start_year, end_year):
    air_pollution_data = []
    for location, (lat, lon) in locations.items():
        print(f"Fetching data for {location}...")
        for year in range(start_year, end_year + 1):
            for month in range(1, 13):
                start_date = f"{year}-{month:02d}-01 00:00"
                last_day = calendar.monthrange(year, month)[1]
                end_date = f"{year}-{month:02d}-{last_day:02d} 23:59"
                start_timestamp = datetime_to_unix(start_date)
                end_timestamp = datetime_to_unix(end_date)
                print(f"Fetching data for {year}-{month:02d}...")
                data = get_air_pollution_data(api_key, lat, lon, start_timestamp, end_timestamp)
                if data:
                    for entry in data.get('list', []):
                        timestamp = datetime.utcfromtimestamp(entry['dt']).strftime('%Y-%m-%d %H:%M:%S')
                        aqi = entry['main']['aqi']
                        components = entry['components']
                        air_pollution_data.append({
                            'Location': location,
                            'Timestamp': timestamp,
                            'AQI': aqi,
                            'CO (μg/m³)': components.get('co'),
                            'NO (μg/m³)': components.get('no'),
                            'NO2 (μg/m³)': components.get('no2'),
                            'O3 (μg/m³)': components.get('o3'),
                            'SO2 (μg/m³)': components.get('so2'),
                            'PM2.5 (μg/m³)': components.get('pm2_5'),
                            'PM10 (μg/m³)': components.get('pm10'),
                            'NH3 (μg/m³)': components.get('nh3')
                        })
                time.sleep(1)
    
    df = pd.DataFrame(air_pollution_data)
    filename = f"AirPollutionData_{start_year}_to_{end_year}.csv"
    df.to_csv(filename, index = False)
    print(f"Data saved to {filename}")

In [ ]:
process_and_save_air_pollution_data(api_key, locations, 2019, 2024)